# 05 survival analysis

First inactivity spell is absorbing for this endpoint, even after revival; censored customers remain in risk sets.

Reusable calculations live in src/ and run_pipeline.py. This notebook reads executed artifacts so analysis cannot silently retrain or leak future data.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
results=json.loads((root/'artifacts/tables/results.json').read_text())

In [2]:
data=pd.read_csv(root/'artifacts/tables/survival_customers.csv')
display(data.head(12))
display(data.describe(include='all').T)

,customer,duration,event,first_month
0,12346,155.190278,1,12
1,12347,351.769444,1,10
2,12348,77.000000,1,9
3,12349,95.859028,1,4
4,12350,77.000000,1,2
5,12351,77.000000,1,11
6,12352,93.990972,1,11
7,12353,77.000000,1,10
8,12354,77.000000,1,4
9,12355,77.000000,1,5


,count,mean,std,min,25%,50%,75%,max
customer,5878.0,15315.313542,1715.572666,12346.000000,13833.25,15314.5,16797.750000,18287.000000
duration,5878.0,126.043073,124.457473,-0.511111,77.00,77.0,123.659375,737.620833
event,5878.0,0.850800,0.356316,0.000000,1.00,1.0,1.000000,1.000000
first_month,5878.0,7.046785,3.831577,1.000000,3.00,7.0,11.000000,12.000000


Interpretation: consult the corresponding report for mathematical definitions, executed estimates and their limits. Never turn an assumed campaign response into an observed result.

# Survival analysis
S(t)=P(T>t), where T is days from first observed purchase until the first completed 77-day purchase gap. The hazard h(t)=lim[Δt→0] P(t≤T<t+Δt | T≥t)/Δt. Kaplan-Meier is the product over event times of (1−d_j/n_j). A subsequent revival does not erase the first event. If no qualifying gap completes by the end of observation, duration ends at the administrative cutoff and event=0.
```json
{
  "First observed Jan-Jun": {
    "customers": 2717,
    "events": 2613,
    "survival_180_days": 0.15393792898397185,
    "median_days": 77.0
  },
  "First observed Jul-Dec": {
    "customers": 3161,
    "events": 2388,
    "survival_180_days": 0.19117101560927535,
    "median_days": 77.0
  }
}
```
Curves include Greenwood confidence intervals. Jan-Jun versus Jul-Dec first-observed season is a baseline segment; eventual lifetime spend would introduce post-event information. First-observed customers are not verified new customers. There is a structural event-free interval before the threshold. Cohort length and seasonality limit group comparisons. Cox PH is deliberately omitted: stationarity, left truncation and first-gap events make an explanatory PH fit insufficiently justified here. In a longer bank panel, fit baseline covariates, inspect Schoenfeld residuals, time interactions and PH assumptions, and interpret hazard ratios associationally.
